# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a line chart to compare population growth across continents from 1970 to 2022. I used 1970 as the base year (100) so that we can compare relative growth fairly, even though continents have different population sizes. Africa experienced the fastest relative population growth. I highlighted Africa in blue and used grey for the other continents to make the comparison easier to understand.

In [2]:

# Q1 - Which continent is growing fastest?

# Step 1: Calculate total population for each continent and year
data = df_long.groupby(
    ['Continent', 'Year'], as_index=False
)['Population'].sum()

# Step 2: Find each continent's population in 1970
baseline = data[data['Year'] == 1970].set_index(
    'Continent'
)['Population']

# Step 3: Set 1970 to 100 for every continent
data['Growth Index'] = (
    data['Population'] /
    data['Continent'].map(baseline) * 100
)

# Step 4: Choose colours
colors = {
    continent: '#AAAAAA'
    for continent in data['Continent'].unique()
}
colors['Africa'] = '#0072B2'

# Step 5: Create the line chart
fig = px.line(
    data,
    x='Year',
    y='Growth Index',
    color='Continent',
    color_discrete_map=colors
)

# Step 6: Highlight Africa
for trace in fig.data:
    trace.update(
        line=dict(width=4 if trace.name == 'Africa' else 2)
    )

# Step 7: Label the end of each line
shifts = {
    'Africa': 0,
    'Oceania': 24,
    'South America': 4,
    'Asia': -17,
    'North America': 0,
    'Europe': 0
}

for _, row in data[data['Year'] == 2022].iterrows():
    fig.add_annotation(
        x=2022,
        y=row['Growth Index'],
        text=row['Continent'],
        showarrow=False,
        xanchor='left',
        xshift=10,
        yshift=shifts[row['Continent']],
        font=dict(
            color=colors[row['Continent']],
            size=12
        )
    )

# Step 8: Make it clean
fig.update_layout(
    title="Africa's population grew fastest since 1970",
    template='plotly_white',
    showlegend=False,
    xaxis_title=None,
    yaxis_title='Population index (1970 = 100)',
    xaxis=dict(range=[1969, 2045], showgrid=False),
    yaxis=dict(showgrid=False),
    margin=dict(r=140)
)

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a sorted horizontal bar chart because it makes it easy to compare the population increases of 15 countries and identify the largest increases. I calculated the difference between the 2022 and 1970 populations. I used blue to highlight African countries and grey for other countries. India had the largest population increase, and 4 of the top 15 countries are in Africa.

In [3]:

# Q2 - Which countries added the most people?

# Calculate population increase from 1970 to 2022
top15 = df[['Country', 'Continent',
            '1970 Population', '2022 Population']].copy()

top15['Added (millions)'] = (
    top15['2022 Population'] - top15['1970 Population']
) / 1_000_000

# Select the top 15 countries and sort them
top15 = top15.nlargest(15, 'Added (millions)')
top15 = top15.sort_values('Added (millions)')

# Count African countries
africa_count = (top15['Continent'] == 'Africa').sum()
print("African countries in top 15:", africa_count)

# Highlight Africa in blue; others in grey
colors = [
    '#0072B2' if c == 'Africa' else '#B7B7B7'
    for c in top15['Continent']
]

# Create horizontal bar chart
fig = px.bar(
    top15,
    x='Added (millions)',
    y='Country',
    orientation='h',
    text='Added (millions)',
    category_orders={'Country': top15['Country'].tolist()}
)

# Style bars and show values
fig.update_traces(
    marker_color=colors,
    texttemplate='%{text:.0f}M',
    textposition='outside',
    cliponaxis=False
)

# Polish chart
fig.update_layout(
    title=(
        "India added the most people from 1970 to 2022"
        f"<br><sup>{africa_count} of the top 15 countries"
        " are African | Blue = Africa</sup>"
    ),
    template='plotly_white',
    showlegend=False,
    height=630,
    xaxis_title='Population increase (millions)',
    yaxis_title=None,
    margin=dict(l=125, r=70, t=90, b=55)
)

fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    range=[0, top15['Added (millions)'].max() * 1.18]
)
fig.update_yaxes(showgrid=False)

fig.show()


African countries in top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a scatter plot because it is useful for showing the relationship between two numerical variables - land area and population. I separated the countries by continent to compare their patterns. I used logarithmic scales because countries vary greatly in size and population. Grey points represent most countries, while blue highlights notable exceptions. There is a general positive relationship, but countries such as Canada and Australia have large land areas and relatively small populations, while Bangladesh has a high population despite its smaller area.

In [4]:

# Q3 - Do bigger countries have more people?

# Step 1: Select countries to highlight
exceptions = [
    'Canada', 'Australia', 'Russia',
    'Bangladesh', 'India'
]

data = df.copy()

# Step 2: Create labels and colour groups
data['Point type'] = data['Country'].apply(
    lambda x: 'Notable exceptions'
    if x in exceptions else 'Other countries'
)

data['Label'] = data['Country'].apply(
    lambda x: x if x in exceptions else ''
)

# Step 3: Create scatter plots by continent
fig = px.scatter(
    data,
    x='Area',
    y='2022 Population',
    facet_col='Continent',
    facet_col_wrap=3,
    color='Point type',
    color_discrete_map={
        'Other countries': '#B0B0B0',
        'Notable exceptions': '#0072B2'
    },
    category_orders={
        'Point type': [
            'Other countries',
            'Notable exceptions'
        ]
    },
    text='Label',
    hover_name='Country',
    log_x=True,
    log_y=True,
    opacity=0.8
)

# Step 4: Make the dots and labels readable
fig.update_traces(
    marker=dict(size=8),
    textposition='top center'
)

# Step 5: Simplify the continent headings
fig.for_each_annotation(
    lambda a: a.update(
        text=a.text.split('=')[-1]
    )
)

# Step 6: Remove unnecessary gridlines
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    title_text=None
)

fig.update_yaxes(
    showgrid=False,
    zeroline=False,
    title_text=None
)

# Step 7: Add shared axis labels
fig.add_annotation(
    x=0.5, y=-0.09,
    xref='paper', yref='paper',
    text='Land area (km², log scale)',
    showarrow=False
)

fig.add_annotation(
    x=-0.08, y=0.5,
    xref='paper', yref='paper',
    text='Population (2022, log scale)',
    textangle=-90,
    showarrow=False
)

# Step 8: Add insight title and finish chart
fig.update_layout(
    title=(
        "Land area alone does not explain population"
        "<br><sup>Canada and Australia contrast"
        " with densely populated Bangladesh</sup>"
    ),
    template='plotly_white',
    showlegend=False,
    height=850,
    margin=dict(l=100, r=40, t=100, b=110)
)

fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a box plot because it shows the spread of population density across countries and territories within each continent, rather than only showing an average. The boxes represent the middle 50% of values, the line inside each box shows the median, and the points beyond the whiskers show outliers. I used a logarithmic scale because population densities vary greatly. North America has the highest median country or territory density, while Asia and Europe contain some extreme high-density outliers. I highlighted North America in blue and used grey for the other continents.

In [5]:

# Q4 - How crowded are the continents?

# Step 1: Prepare the data
data = df[df['Density'] > 0].copy()

# Step 2: Calculate median density for each continent
medians = data.groupby('Continent')['Density'].median()
medians = medians.sort_values(ascending=False)

# Step 3: Highlight the continent with the highest median
highest = medians.index[0]

data['Highlight'] = np.where(
    data['Continent'] == highest,
    'Highest median',
    'Other continents'
)

# Step 4: Create the box plot
fig = px.box(
    data,
    x='Density',
    y='Continent',
    color='Highlight',
    orientation='h',
    points='outliers',
    log_x=True,
    hover_name='Country',
    category_orders={
        'Continent': medians.index.tolist()
    },
    color_discrete_map={
        'Highest median': '#0072B2',
        'Other continents': '#AAAAAA'
    }
)

# Step 5: Style the boxes and points
fig.update_traces(
    marker=dict(size=7, opacity=0.75),
    line=dict(width=1.5)
)

# Step 6: Label two extreme outliers
for country in ['Macau', 'Monaco']:
    row = data[data['Country'] == country].iloc[0]

    fig.add_annotation(
        x=float(row['Density']),
        y=row['Continent'],
        text=country,
        showarrow=True,
        ax=-55,
        ay=-28,
        arrowcolor='#777777'
    )

# Step 7: Add title and clean up the chart
fig.update_layout(
    title=(
        "North America has the highest median density;"
        "<br><sup>Asia and Europe have extreme outliers</sup>"
    ),
    template='plotly_white',
    showlegend=False,
    height=570,
    xaxis_title='People per km² (log scale)',
    yaxis_title=None,
    margin=dict(l=120, r=80, t=90, b=65)
)

fig.update_xaxes(
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    showgrid=False,
    categoryorder='array',
    categoryarray=medians.index.tolist()[::-1]
)

# Step 8: Display the chart
fig.show()

# Show the median densities
print("Median density by continent:")
print(medians.round(1))


Median density by continent:
Continent
North America    221.4
Asia             123.0
Europe           106.9
Oceania           78.2
Africa            59.7
South America     17.8
Name: Density, dtype: float64


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a choropleth map because it clearly shows the geographical distribution of population growth and decline between 2000 and 2022. I calculated the percentage change for each country and territory. Blue represents population growth, orange represents population decline, and grey represents little or no change. The map shows that most countries experienced population growth, while declines occurred in parts of Europe and several island territories. The colour intensity helps compare the magnitude of population change.

In [6]:

# Q5 - Where did populations grow and shrink?

# Step 1: Make a copy of the dataset
data = df.copy()

# Step 2: Calculate percentage population change
data['Change (%)'] = (
    (data['2022 Population'] - data['2000 Population'])
    / data['2000 Population']
) * 100

# Step 3: Count countries with growth and decline
grew = (data['Change (%)'] > 0).sum()
shrank = (data['Change (%)'] < 0).sum()

print("Countries and territories that grew:", grew)
print("Countries and territories that shrank:", shrank)

# Step 4: Set the colours
# Orange = decline, grey = no change, blue = growth
color_scale = [
    (0.00, '#C47A19'),
    (0.06, '#E5A436'),
    (0.1111, '#E8E8E8'),
    (0.18, '#B8D8E8'),
    (0.50, '#438FC0'),
    (1.00, '#0072B2')
]

# Step 5: Create the world map
fig = px.choropleth(
    data,
    locations='CCA3',
    locationmode='ISO-3',
    color='Change (%)',
    hover_name='Country',
    hover_data={
        'Continent': True,
        '2000 Population': ':,',
        '2022 Population': ':,',
        'Change (%)': ':.1f',
        'CCA3': False
    },
    color_continuous_scale=color_scale,
    range_color=(-40, 320),
    projection='natural earth'
)

# Step 6: Add an insight title
fig.update_layout(
    title=(
        f"Population grew in {grew} countries and territories"
        f" and shrank in {shrank}"
        "<br><sup>Population change, 2000–2022"
        " | Orange = decline, Blue = growth</sup>"
    ),
    template='plotly_white',
    height=600,
    margin=dict(l=10, r=10, t=100, b=5),
    font=dict(size=13)
)

# Step 7: Make the map clean and readable
fig.update_geos(
    showframe=False,
    showcoastlines=False,
    showcountries=True,
    countrycolor='white',
    showland=True,
    landcolor='#EAEAEA',
    showocean=True,
    oceancolor='#F8F8F8'
)

# Step 8: Format the colour legend
fig.update_layout(
    coloraxis_colorbar=dict(
        title='Change (%)',
        tickvals=[-40, 0, 50, 100, 200, 300],
        ticksuffix='%'
    )
)

# Step 9: Display the map
fig.show()


Countries and territories that grew: 200
Countries and territories that shrank: 34


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a treemap because it shows two levels of population distribution in one figure: continents and the countries within them. The size of each rectangle represents its population in 2022, making it easy to compare their relative shares. Asia contains the largest share of the world's population, approximately 59.2%, followed by Africa. I highlighted Asia in blue and used grey for the other continents to emphasize this finding. The hierarchical layout helps readers see both the overall continental distribution and the contribution of individual countries.

In [7]:

# Q6 - Where do the world's 8 billion people live?

# Step 1: Prepare the data
data = df.copy()

# Step 2: Calculate each continent's population
continent_pop = data.groupby('Continent')[
    '2022 Population'
].sum().sort_values(ascending=False)

world_total = data['2022 Population'].sum()
asia_share = (
    continent_pop['Asia'] / world_total * 100
)

print("World population:", f"{world_total:,}")
print("Asia's share:", f"{asia_share:.1f}%")

# Step 3: Highlight Asia in blue
data['Highlight'] = np.where(
    data['Continent'] == 'Asia',
    'Asia',
    'Other continents'
)

# Step 4: Create the treemap
fig = px.treemap(
    data,
    path=[
        px.Constant('World'),
        'Continent',
        'Country'
    ],
    values='2022 Population',
    color='Highlight',
    color_discrete_map={
        'Asia': '#0072B2',
        'Other continents': '#B4B4B4'
    }
)

# Step 5: Show useful labels and hover details
fig.update_traces(
    texttemplate=(
        '<b>%{label}</b><br>%{percentRoot:.1%}'
    ),
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,.0f}<br>'
        'World share: %{percentRoot:.1%}'
        '<extra></extra>'
    ),
    marker=dict(
        line=dict(color='white', width=1.5)
    )
)

# Step 6: Make text readable
fig.update_layout(
    uniformtext=dict(
        minsize=11,
        mode='hide'
    )
)

# Step 7: Add the insight title and clean layout
fig.update_layout(
    title=(
        f"Asia holds {asia_share:.1f}% of the world's"
        " population in 2022"
        "<br><sup>Rectangle size = population"
        " | Blue = Asia</sup>"
    ),
    template='plotly_white',
    showlegend=False,
    height=680,
    margin=dict(l=15, r=15, t=90, b=15)
)

# Step 8: Display the treemap
fig.show()


World population: 7,973,413,042
Asia's share: 59.2%


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a 100% stacked bar chart because it shows how each region's percentage share of the world population changed over time, while keeping every year's total at 100%. I grouped North America, South America and Oceania as "Rest of the world". I highlighted Africa in blue to emphasize its increasing share and used neutral grey shades for the other regions. The chart shows that Africa's share rose from 9.9% in 1970 to 17.9% in 2022, while Europe's share fell from 17.8% to 9.3%. Asia remained the largest region, with a share of approximately 59% in 2022.

In [8]:

# Q7 - Whose share of the world is rising?

# Step 1: Calculate population by continent and year
continent_year = df_long.groupby(
    ['Year', 'Continent'], as_index=False
)['Population'].sum()

# Step 2: Group the other three continents together
continent_year['Group'] = continent_year['Continent'].where(
    continent_year['Continent'].isin(
        ['Africa', 'Asia', 'Europe']
    ),
    'Rest of the world'
)

# Step 3: Calculate population for each group
shares = continent_year.groupby(
    ['Year', 'Group'], as_index=False
)['Population'].sum()

# Step 4: Convert population into percentage shares
shares['Share (%)'] = (
    shares['Population'] /
    shares.groupby('Year')['Population'].transform('sum')
    * 100
)

# Step 5: Prepare years and labels
shares['Year label'] = shares['Year'].astype(str)

shares['Text'] = np.where(
    shares['Year'].isin([1970, 2022]),
    shares['Share (%)'].map(lambda n: f'{n:.1f}%'),
    ''
)

# Step 6: Create a 100% stacked bar chart
fig = px.bar(
    shares,
    x='Year label',
    y='Share (%)',
    color='Group',
    text='Text',
    category_orders={
        'Year label': [
            str(y) for y in sorted(shares['Year'].unique())
        ],
        'Group': [
            'Rest of the world',
            'Europe',
            'Asia',
            'Africa'
        ]
    },
    color_discrete_map={
        'Rest of the world': '#C3CBD1',
        'Europe': '#8D989F',
        'Asia': '#4F5964',
        'Africa': '#0072B2'
    }
)

# Step 7: Make labels readable
fig.update_traces(
    textposition='inside',
    insidetextanchor='middle',
    marker_line_width=0,
    hovertemplate=(
        'Year: %{x}<br>'
        'Share: %{y:.1f}%'
        '<extra>%{fullData.name}</extra>'
    )
)

for trace in fig.data:
    if trace.name in ['Rest of the world', 'Europe']:
        trace.textfont.color = '#1D2930'
    else:
        trace.textfont.color = 'white'

# Step 8: Add title and clean layout
fig.update_layout(
    barmode='stack',
    title=(
        "Africa's share of the world has nearly doubled"
        " since 1970"
        "<br><sup>Europe's share has nearly halved"
        " | Each bar totals 100%</sup>"
    ),
    template='plotly_white',
    height=580,
    legend_title_text=None,
    legend=dict(
        orientation='h',
        y=-0.18,
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=60, r=25, t=90, b=100),
    xaxis_title=None,
    yaxis_title='Share of world population'
)

# Step 9: Format axes
fig.update_yaxes(
    range=[0, 100],
    ticksuffix='%',
    showgrid=False,
    zeroline=False
)

fig.update_xaxes(showgrid=False)

# Step 10: Display the chart
fig.show()

# Print values for checking
print(
    shares.pivot(
        index='Group',
        columns='Year',
        values='Share (%)'
    )[[1970, 2022]].round(1)
)


Year               1970  2022
Group                        
Africa              9.9  17.9
Asia               58.1  59.2
Europe             17.8   9.3
Rest of the world  14.3  13.6
